# Sky Segmentation - Colab 訓練與評估範例

本 notebook 示範如何在 Google Colab 上執行完整的訓練和評估流程。

## 1. 安裝依賴

In [ ]:
# 安裝必要的套件
!pip install torch torchvision transformers timm tqdm pillow numpy pandas

## 2. 克隆 GitHub Repository

In [ ]:
# 替換為您的 GitHub repository URL
!git clone https://github.com/your-username/your-repo.git
%cd your-repo

## 3. 準備數據

In [ ]:
# 方法1：從 Google Drive 掛載數據
from google.colab import drive
drive.mount('/content/drive')

# 複製數據到 Colab 環境（或直接使用 Drive 路徑）
# !cp -r /content/drive/MyDrive/your_data_folder /content/data

In [ ]:
# 方法2：直接上傳數據（適合小數據集）
# 使用 Colab 的文件上傳功能

## 4. 建立數據 Splits

In [ ]:
# 步驟 1：建立 metadata
!python colab/create_splits.py --step metadata \
    --data_dir /content/data \
    --outputs_dir /content/outputs

In [ ]:
# 步驟 2：建立 train/val/test splits
!python colab/create_splits.py --step splits \
    --data_dir /content/data \
    --outputs_dir /content/outputs \
    --metadata_file /content/outputs/metadata_all_images.csv

## 5. 訓練模型

In [ ]:
# 訓練所有模型（預設使用 GPU）
!python colab/train_all.py \
    --data_dir /content/data \
    --outputs_dir /content/outputs \
    --batch_size 8 \
    --image_size 512 512 \
    --num_workers 2 \
    --device cuda

In [ ]:
# 或只訓練特定模型
!python colab/train_all.py --models dl,dino-mlp \
    --data_dir /content/data \
    --outputs_dir /content/outputs \
    --batch_size 8 \
    --image_size 512 512 \
    --device cuda

## 6. 評估模型

In [ ]:
# 評估所有模型（包含 overlay 輸出）
!python colab/eval_all.py \
    --data_dir /content/data \
    --outputs_dir /content/outputs \
    --image_size 512 512 \
    --overlay_dir /content/outputs/overlays \
    --device cuda

In [ ]:
# 或只評估特定模型
!python colab/eval_all.py --models dl,sam \
    --data_dir /content/data \
    --outputs_dir /content/outputs \
    --overlay_dir /content/outputs/overlays \
    --device cuda

## 7. 查看結果

In [ ]:
# 讀取 metrics 摘要
import pandas as pd

metrics_df = pd.read_csv('/content/outputs/metrics_summary.csv')
print(metrics_df)

In [ ]:
# 顯示 overlay 影像（範例）
from IPython.display import Image, display
import os

overlay_dir = '/content/outputs/overlays'
if os.path.exists(overlay_dir):
    overlay_files = [f for f in os.listdir(overlay_dir) if f.endswith('.png')][:5]
    for f in overlay_files:
        display(Image(os.path.join(overlay_dir, f)))

## 8. 下載結果到本地

In [ ]:
# 方法1：下載到本地
from google.colab import files

# 下載 metrics CSV
files.download('/content/outputs/metrics_summary.csv')

# 下載 checkpoints（可選）
# !zip -r checkpoints.zip /content/outputs/train_*/
# files.download('checkpoints.zip')

In [ ]:
# 方法2：保存到 Google Drive
# !cp -r /content/outputs /content/drive/MyDrive/